# Week 4 follow-up

This note starts after [`00_briefing.ipynb`](00_briefing.ipynb). Same ten downtown boxes, same `resi_def_v1` labels, same leave-one-metro-out split. The question here is whether a small embedding of a 100 m tile separates residential from non-residential inside each metro.

Two summaries. Linear models on AlphaEarth already sit near 0.76 LORO macro-F1 at 100 m, and PLS does not beat the 64-D logistic regression. Forcing the ViT down to 3 numbers raised the held-out gap and left F1 about where v1 and v3 were. Atlanta, Phoenix, and Denver clear a held-out gap of 3. New York does not.

| Run | LORO macro-F1 | Mean held-out gap |
|---|---:|---:|
| 100 m mean-pool logistic regression | 0.761 | — |
| ViT, 128-D class token | 0.756 | — |
| v1, BCE + SupCon | 0.732 | 1.76 |
| v2, detached head | 0.632 | 2.12 |
| v3, same loss, longer training | 0.735 | 1.71 |
| v4, within-region pull and gap | 0.738 | 2.13 |

Gap = distance between the two class centers ÷ mean radius of each class. A gap of 3 was the training target. A gap near 1 is one cloud. Raw sep is not comparable across runs: v4's clouds are much smaller than v1's, and the ratio is the number to compare. Each fold has its own 3D coordinates.


## Linear baselines

100 m features are the mean of the AlphaEarth cells inside the tile. Tiles need at least 5 occupied cells, then residential and non-residential are balanced (n=52,492). The 10 m numbers use the balanced pure cells (n=1,751,814).

| Scale | Model | Width | LORO macro-F1 |
|---|---|---:|---:|
| 10 m cells | 64-D logistic regression | 64 | 0.779 ± 0.085 |
| 10 m cells | PLS-DA + logistic regression | 32 | 0.775 ± 0.088 |
| 10 m cells | PCA + logistic regression | 32 | 0.759 ± 0.084 |
| 10 m cells | L1 on 32 PLS scores | 32, C=30 | 0.751 ± 0.105 |
| 10 m cells | t1 only | 1 | 0.735 ± 0.117 |
| 100 m mean pool | 64-D logistic regression | 64 | 0.761 ± 0.092 |
| 100 m mean pool | PLS-DA + logistic regression | 24 | 0.762 ± 0.096 |
| 100 m mean pool | PCA + logistic regression | 32 | 0.749 ± 0.090 |
| 100 m mean pool | L1 on 32 PLS scores | 32, C=30 | 0.738 ± 0.098 |
| 100 m mean pool | t1 only | 1 | 0.720 ± 0.119 |
| 100 m | flat 10×10 grid, logistic regression | 6500 | 0.700 |
| 100 m | ViT, 128-D class token | 128 | 0.756 ± 0.119 |

Moving from 10 m cells to a 100 m mean costs 0.018 F1 on the 64-D logistic regression (0.779 → 0.761) and 0.013 on the best PLS-DA head. PLS-DA at 100 m, k=24, is 0.762, level with the full 64-D head. PCA is lower. The first PLS score alone is 0.720.

L1 on the 32 PLS scores drops the mean number of nonzero weights from 32 to about 1.2 at C=10, and that survivor is t1 on every fold where anything survives (t1 survival 1.0, t2 survival 0.2). LORO F1 at that setting is 0.735, the same neighborhood as the t1-only head. Stronger L1 zeros every score.

k=9 is the smallest PLS width within 0.02 F1 of the 64-D logistic regression (F1 0.761, gap 0.018). The ViT reads those 9 scores. Empty cells in the 10×10 grid are masked.

![LORO F1 at 10 m and 100 m](../../outputs/week4_pls_10m_vs_100m/figures/f1_family_10m_vs_100m.png)

![Which PLS scores survive L1](../../outputs/week4_l1_pls_loro/figures/l1_survival_pls32.png)

![ViT against the linear tile heads](../../outputs/week4_vit_100m/figures/vit_vs_baselines.png)

Notebooks: [`08_pls_comparison.ipynb`](08_pls_comparison.ipynb), [`09_l1_pls_sparsity.ipynb`](09_l1_pls_sparsity.ipynb), [`10_pls_10m_vs_100m.ipynb`](10_pls_10m_vs_100m.ipynb).


## Why the 3D runs changed

The 128-D ViT matches the mean-pool logistic regression and does not show two clusters, because there is no 3D space to look at. The next runs keep the same ViT (d_model 128, 4 layers, 4 heads) and replace the class token with 3 numbers. The aim is two residential / non-residential clusters inside each metro after fine-tuning.

v1 adds supervised contrastive loss on the L2-normalized 3D vector, λ=1, temperature 0.1, and lets the classifier gradient into the encoder. Held-out F1 is 0.732. The clouds are horseshoes: the linear head only needs one direction, so the extra two axes collapse into a curve. The tips of the curve are class-pure. The bottom is mixed. SupCon's minimum for two classes is two opposite poles. This run did not reach that minimum.

v2 makes four changes: stop the classifier gradient at the 3D vector, use a cosine head with a learnable scale, set λ=10, and anneal temperature from 0.5 to 0.1. The encoder's only training signal is SupCon. Held-out F1 falls to 0.632. SupCon stays near the chance value for a batch of 256 (about log 255 ≈ 5.54). Atlanta is the clear failure: the checkpoint is epoch 1, training stops at epoch 8, and every held-out tile is predicted non-residential (F1 0.333). v1 Atlanta was 0.860. The stop rule was validation F1, while training BCE was still falling.

v3 keeps the v2 loss and changes the schedule. Every fold runs at least 35 epochs, then stops on validation loss (BCE + λ τ SupCon), patience 6, cap 60. F1 is logged and does not choose the checkpoint. Held-out F1 returns to 0.735, in line with v1. All 10 folds run to the cap. Temperature is still annealing across the 60-epoch budget, so the weighted validation loss keeps falling while validation SupCon moves back toward chance. Longer training did not make two poles.

v4 drops SupCon, the detach, and the temperature. Each batch is one metro, half residential and half non-residential. On the raw 3D vectors:

```text
loss = BCE(scale * (z · unit w), y)
     + mean squared distance to the class mean
     + relu(3 − sep/within)^2
```

Both the classifier and the cluster terms send gradient into z. The stop metric is the mean sep/within on a 10% split of the other metros. Higher is better. The floor is still 35 epochs, patience 6, cap 60. F1 does not choose the checkpoint. Held-out F1 is 0.738. Mean held-out gap is 2.13, compared with 1.71 in v3.

Notebooks: [`12_vit_100m_3d.ipynb`](12_vit_100m_3d.ipynb) through [`15_vit_100m_3d_v4.ipynb`](15_vit_100m_3d_v4.ipynb).


## Held-out metros under v4

Atlanta, Phoenix, and Denver are the held-out metros with gap ≥ 3. The gap is higher than v3 in every metro. F1 moves by metro and the mean is flat (0.738 vs v3 0.735). Chicago drops from 0.622 to 0.553. New York stays one cloud: gap 0.63, F1 0.586.

The validation column is the mean gap on a 10% split of the other metros. That is the number training stopped on. New York's fold reached a validation gap of 3.33 and a held-out gap of 0.63.

| Metro | v4 F1 | v3 F1 | v1 F1 | v4 gap | v3 gap | v4 val gap | epochs |
|---|---:|---:|---:|---:|---:|---:|---:|
| Atlanta | 0.898 | 0.840 | 0.860 | 3.51 | 2.34 | 2.69 | 35 |
| Phoenix | 0.878 | 0.849 | 0.888 | 3.19 | 2.43 | 2.75 | 40 |
| Denver | 0.862 | 0.852 | 0.867 | 3.15 | 2.81 | 2.85 | 42 |
| Seattle | 0.824 | 0.787 | 0.788 | 2.81 | 2.29 | 2.95 | 45 |
| Los Angeles | 0.793 | 0.842 | 0.786 | 2.50 | 2.39 | 3.00 | 35 |
| Houston | 0.709 | 0.711 | 0.693 | 1.96 | 1.65 | 2.94 | 41 |
| Chicago | 0.553 | 0.622 | 0.610 | 1.32 | 1.09 | 3.07 | 35 |
| Boston | 0.693 | 0.698 | 0.702 | 1.30 | 1.01 | 3.13 | 45 |
| San Francisco | 0.584 | 0.595 | 0.573 | 0.99 | 0.76 | 3.06 | 35 |
| New York | 0.586 | 0.552 | 0.554 | 0.63 | 0.35 | 3.33 | 35 |

![Held-out gap, v4 against v3](../../outputs/week4_vit_100m_3d_v4/figures/v4_vs_v3_gap.png)


## How to read a cluster figure

Left panel: the three encoder outputs, unrotated. The gray plane is the classifier cut, `z · w = 0`. Right panel: the same tiles on the classifier score and the widest direction perpendicular to it. The dashed line is the same cut. Residential is on the positive side of the score.

Color is the mix of the two class centers. Orange is non-residential, blue is residential, mauve is where both centers still reach. Influence of a center falls with distance, and the radius reaches at least halfway to the other center, so the overlap is a blend instead of one class drawn on top of the other. Marker size falls off with that influence. The white-edged dots are the two centers.

A high gap does not mean an empty band between the classes. Denver's held-out gap is 3.15 and its F1 is 0.862. 89% of non-residential tiles have score below 0, and 83% of residential tiles have score above 0. Along the score, the leftmost 30% of tiles are 9% residential, the middle 40% are 46% residential, and the rightmost 30% are 96% residential. Almost every tile of either class still sits inside the other class's score range (over 99.9% of each). The cut sorts the mass. The supports overlap.

The geometry is a thin line along the class axis. A different camera angle does not open a gap.

Atlanta clears the gap. Denver is the high-F1 case above. New York is the held-out cloud.

![Atlanta, held out](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Atlanta_SE.png)

![Denver, held out](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_Denver_W.png)

![New York, held out](../../outputs/week4_vit_100m_3d_v4/figures/vit3d_NYC_NE.png)


## Validation while New York is held out

The New York fold's best validation gap is 3.33, measured on 10% of the other metros at the checkpoint (logged best 3.334). The same checkpoint's held-out New York gap is 0.63. The other cities in that fold do separate. They are still thin lines with the classes at opposite ends.

| Metro in the New York fold's validation split | Gap | Sep | Within | n |
|---|---:|---:|---:|---:|
| Phoenix | 5.44 | 0.553 | 0.102 | 514 |
| Atlanta | 4.98 | 0.458 | 0.092 | 367 |
| Denver | 3.74 | 0.468 | 0.125 | 743 |
| Los Angeles | 3.71 | 0.440 | 0.118 | 1068 |
| Seattle | 3.59 | 0.396 | 0.110 | 225 |
| Houston | 2.80 | 0.324 | 0.116 | 637 |
| Chicago | 2.01 | 0.264 | 0.131 | 505 |
| Boston | 1.90 | 0.261 | 0.138 | 550 |
| San Francisco | 1.82 | 0.173 | 0.095 | 111 |

![Phoenix in the New York fold's validation split](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_Phoenix_SW.png)

![San Francisco in the New York fold's validation split](../../outputs/week4_vit_100m_3d_v4/figures/val_nycfold_SF_W.png)

The other held-out figures, and the rest of this validation split, are in [`08_pls_comparison.ipynb`](08_pls_comparison.ipynb).


## Paths

| Artifact | Path |
|---|---|
| PLS LORO | `outputs/week4_pls_loro/` |
| 10 m vs 100 m | `outputs/week4_pls_10m_vs_100m/` |
| L1 on PLS scores | `outputs/week4_l1_pls_loro/` |
| ViT 128-D | `outputs/week4_vit_100m/` |
| 3D v1–v3 | `outputs/week4_vit_100m_3d/`, `_v2/`, `_v3/` |
| 3D v4 folds, checkpoints, figures | `outputs/week4_vit_100m_3d_v4/` |
| This note | `notebooks/week4/16_briefing.ipynb` |
